# ddldelta: the operational workflow, snapshots and the CLI

The first notebook diffed two DDL directories directly. In a real pipeline
there is usually no second directory lying around to diff against — a
supplier delivery gets processed and then archived, or the source is a
one-shot export. `ddldelta`'s answer to "diff against what, then?" is a
**snapshot**: a small JSON file that remembers the parsed shape of the
schema after the last successful run, so the next run has something to
diff against without keeping old DDL trees around.

This notebook covers the operational pieces built around that idea:

1. running `ddldelta generate` through the command line (in-process, no
   shell) for an initial delivery, storing a snapshot,
2. a second delivery diffed against that snapshot,
3. `ddldelta check` as a CI gate, with its `0`/`1`/`2` exit-code contract,
4. `ddldelta baseline` for a fresh environment,
5. a one-cell look at the Flyway renderer,
6. library-only knobs the CLI does not expose.

In [1]:
import tempfile
from pathlib import Path

work = Path(tempfile.mkdtemp(prefix="ddldelta_tutorial_"))
snapshot_path = work / "product_state.json"
migrations_dir = work / "migrations"
print("workspace:", work.name)

workspace: ddldelta_tutorial_03clndt0


In [2]:
import io
from contextlib import redirect_stdout

from ddldelta.cli import main


def run_cli(argv):
    """Run the ddldelta CLI in-process and return (rc, output) with the
    workspace's own absolute path stripped down to a relative one, so the
    printed output stays readable and free of local filesystem detail."""
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        rc = main(argv)
    text = buffer.getvalue().replace(str(work) + "\\", "").replace(str(work), ".")
    return rc, text

## Delivery v1

A `product` table, as a first delivery.

In [3]:
v1_dir = work / "v1"
v1_dir.mkdir()
(v1_dir / "product.sql").write_text("""\
CREATE TABLE `product` (
  `id` int(10) unsigned NOT NULL,
  `sku` varchar(50) NOT NULL,
  `name` varchar(200) NOT NULL,
  PRIMARY KEY (`id`)
);
""", encoding="utf-8")
print(sorted(str(p.relative_to(work)) for p in v1_dir.glob("*.sql")))

['v1\\product.sql']


## First run: `ddldelta generate`, in-process

No `--old` means an initial migration — every table is new. `--snapshot`
tells `generate` to persist the parsed *new* state to that file after a
successful render, so a later run can point `--old` at it instead of an old
DDL directory.

In [4]:
rc, out = run_cli([
    "generate", "--dialect", "mysql", "--label", "1",
    "--out", str(migrations_dir), "--snapshot", str(snapshot_path),
    str(v1_dir),
])
print("rc:", rc)
print(out)

rc: 0
migrations\V1.0001__product.sql
product_state.json



## Delivery v2: add a column

`product` gains a nullable `description` column — safe. This time `--old`
points at the snapshot instead of a `v1` directory, and `--snapshot` is
passed again to roll the stored state forward to v2.

In [5]:
v2_dir = work / "v2"
v2_dir.mkdir()
(v2_dir / "product.sql").write_text("""\
CREATE TABLE `product` (
  `id` int(10) unsigned NOT NULL,
  `sku` varchar(50) NOT NULL,
  `name` varchar(200) NOT NULL,
  `description` varchar(500) DEFAULT NULL,
  PRIMARY KEY (`id`)
);
""", encoding="utf-8")

rc, out = run_cli([
    "generate", "--dialect", "mysql", "--label", "2",
    "--out", str(migrations_dir),
    "--old", str(snapshot_path), "--snapshot", str(snapshot_path),
    str(v2_dir),
])
print("rc:", rc)
print(out)

rc: 0
migrations\V2.0001__product.sql
product_state.json



Only the v2 delta file is printed above — the v1 file already exists on
disk with identical facts, so the exists-guard silently skips it instead of
rewriting a file schemachange may already have checksummed.

## `ddldelta check`: the CI gate, and its exit codes

`check` prints a compatibility report and exits `0` when the new state is
backward compatible, `1` when it found a critical change (an unrelated
error, e.g. a bad path or dialect, is always `2` — never `1`, so a crash can
never be mistaken for a deliberate finding by a script reading only the
exit code).

To see both outcomes, `v3` drops the `sku` column — a critical change.

In [6]:
v3_dir = work / "v3"
v3_dir.mkdir()
(v3_dir / "product.sql").write_text("""\
CREATE TABLE `product` (
  `id` int(10) unsigned NOT NULL,
  `name` varchar(200) NOT NULL,
  `description` varchar(500) DEFAULT NULL,
  PRIMARY KEY (`id`)
);
""", encoding="utf-8")

rc_ok, out_ok = run_cli(["check", "--dialect", "mysql", str(snapshot_path), str(v2_dir)])
print("v2 vs stored snapshot -> rc:", rc_ok)
print(out_ok)

rc_bad, out_bad = run_cli(["check", "--dialect", "mysql", str(snapshot_path), str(v3_dir), "--json"])
print("v3 vs stored snapshot -> rc:", rc_bad)
print(out_bad)

v2 vs stored snapshot -> rc: 0
2 -> v2: no schema changes

v3 vs stored snapshot -> rc: 1
{
  "comparison": "2 -> v3",
  "compatible": false,
  "exit_code": 1,
  "findings": [
    {
      "table": "product",
      "severity": "critical",
      "statement": "ALTER TABLE \"product\" DROP COLUMN \"sku\";"
    }
  ]
}



The first check compares the snapshot (v2's state, stored a moment ago)
against the `v2` directory itself: identical schema, `rc=0`, "no schema
changes". The second compares it against `v3`'s dropped column: `rc=1`, and
`--json` gives the same finding in a machine-readable shape a CI step can
parse without scraping text.

## `ddldelta baseline`: a fresh environment

`baseline` renders the *current* state as a CREATE-only migration set — no
`--old`, nothing to diff against. Reach for it to stand up a new
environment from scratch, or to re-baseline schemachange/Flyway after
squashing old migrations into one starting point. A baseline can never be
critical (`CreateTable` is always `SAFE`), so there is no fuse to worry
about here.

In [7]:
baseline_dir = work / "baseline"
rc, out = run_cli([
    "baseline", "--dialect", "mysql", "--label", "1",
    "--out", str(baseline_dir), str(v3_dir),
])
print("rc:", rc)
print(out)

baseline_file = next(baseline_dir.glob("V1.*__product.sql"))
print(baseline_file.read_text(encoding="utf-8"))

rc: 0
baseline\V1.0001__product.sql

-- generated by ddldelta ((baseline) -> 1); do not edit manually
CREATE TABLE "product" (
    "id" INT NOT NULL,
    "name" VARCHAR(200) NOT NULL,
    "description" VARCHAR(500)
);



## A Flyway teaser

The library side renders the exact same `MigrationPlan` through
`FlywayRenderer` instead — same file mechanics (fuse, exists-guard,
versioned naming), but it validates that the label is a Flyway-legal
numeric version first. The CLI exposes this as `generate --flyway` /
`baseline --flyway`; here it is the plain library call.

In [8]:
from ddldelta.ddl_parser import parser_for
from ddldelta.sources import parse_path
from ddldelta.plan import baseline_plan
from ddldelta.render.flyway import FlywayRenderer

parse_ddl = parser_for("mysql")
schema = parse_path(parse_ddl, v3_dir)
plan = baseline_plan("1", schema)
flyway_dir = work / "flyway"
written = FlywayRenderer(target_dir=flyway_dir, generated_by="tutorial").render(plan)
print(sorted(str(p.relative_to(work)) for p in written))

['flyway\\V1.0001__product.sql']


## Library-only knobs

The CLI intentionally keeps a few things library-only, reachable through
`GeneratorConfig`/direct calls rather than a flag:

- a custom `TypeMapping` for a dialect/target combination other than the
  built-in Snowflake mapping,
- a custom `ChangePolicy` (e.g. a lenient policy that never fuses, for a
  throwaway dev schema),
- a custom provenance template, for a consumer with an existing,
  differently-worded corpus of deployed files,
- `generate_supplier`'s `GeneratorConfig` orchestration (multi-supplier
  config, `schemachange-config.yml` generation), and `SnapshotSource`'s
  `require_snapshot=True` opt-in for unattended pipelines.

See the README's "Library-only knobs" and "Architecture & extension points"
sections, and `docs/decisions.md` for the reasoning behind each one.